# Human-in-the-Loop

**Module 11 · Lesson 11.10**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Approval Gates: The Risk Boundary
- Confidence Routing: Auto When Sure, Escalate When Not
- The Interrupt/Resume Primitive: The Re-Run Trap
- Escalation: Route to the Right Human, With Context
- Escalation Fatigue: The Silent Killer
- Progressive Autonomy: Earn Trust, Revoke It Instantly

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


## Sign off, or rubber-stamp?

> 💡 **Info**
>
> Your support agent is about to process a **Rs 50,000 refund**. It is roughly 95 percent sure the customer qualifies. Do you let it fire, or does a human sign off first? Now flip the instinct: to be safe, you route *every* action to a human for approval - and within a week your reviewers, drowning in a thousand approvals a day, are clicking “approve” without reading. **Both extremes fail.** Full autonomy ships an irreversible mistake; full review collapses into rubber-stamping. The craft of human-in-the-loop is drawing the line precisely: put a human on the actions that are **irreversible or uncertain**, and let the agent auto-run the rest. This lesson builds every pattern that draws that line - and, unusually for an agents module, you can run almost all of it with no LLM key, because oversight is decision logic.

### 🎯 What you will be able to do after this lesson

- **Build** an approval gate and a confidence router that pause an agent on irreversible or uncertain actions and auto-execute the rest.

- **Compare** the framework HITL primitives (OpenAI needsApproval / Anthropic canUseTool / LangGraph interrupt / Temporal signals) and pick the right one.

- **Operate** the durable interrupt/resume primitive correctly (the node re-runs, so pre-interrupt side effects must be idempotent) and escalate with context.

- **Evaluate** escalation fatigue and progressive autonomy, and know where HITL is legally required (EU AI Act Article 14).

> 📦 **Info**
>
> ✅ Before you startThis builds on **11.6** (computer-use agents gated irreversible clicks behind approval), **11.9** (idempotency and durable resume - which the interrupt primitive here depends on), and **11.5** (the critic pattern - oversight, but by an agent; here the reviewer is a human and the cost is their attention). This lesson is the **oversight** layer; evaluating and monitoring agents is Lesson 11.11.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🤝 **Analogy**
>
> Human-in-the-loop is **a co-pilot, not a chauffeur and not a backseat driver**. A chauffeur (full autonomy) drives while you sleep - wonderful until it takes a wrong turn off a cliff. A backseat driver (full review) makes you approve every lane change until you tune them out entirely. A good co-pilot flies the routine legs and taps you on the shoulder for exactly the decisions that need a human: the irreversible and the uncertain. **Where it breaks down:** a co-pilot knows when to tap you; an agent only knows if you *built* the gate, the confidence check, and the escalation in - oversight is a design decision, never a default.

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“Just review everything to be safe.”** A reviewer flooded with approvals rubber-stamps them - the safety net does not just weaken, it inverts, because now a human has “signed off” on mistakes nobody read.
> - **“interrupt() resumes from where it paused.”** No - the whole node re-runs and interrupt() returns the value on the second pass, so a side effect before it fires *twice* unless it is idempotent.

> 💡 **Info**
>
> ⚠️ Anti-patternTrusting a raw logprob as your confidence signal, and granting full autonomy the moment an agent “works.” Logprobs are systematically overconfident, so an unsure answer still scores high and skips the human it needed; and autonomy granted all at once, with no way to revoke it, means the first bad day is also the worst one. Use ensemble agreement for confidence, and earn autonomy in reversible stages.

---

## 🎯 Concept 1: Approval Gates: The Risk Boundary

### Approval Gates: The Risk Boundary

Read-only actions auto-execute; actions that spend money, send messages, or delete data pause for a human to sign off. The list of risky actions IS the safety mechanism.

Start with the foundational move, the one every other pattern refines: an **approval gate**. Before the agent executes any proposed action, classify it by **risk**. Read-only actions - a search, a lookup, a summary - are safe, so they **auto-execute**. Actions that are **irreversible** - spend money, send a message, delete a record, place an order - **pause and wait for a human to approve or reject**. That set of irreversible actions is not a detail; it *is* the safety mechanism, the boundary between what the agent may do alone and what needs a person. Get the boundary right and most actions never wait; only the ones that could actually hurt do. The block below runs a gate over four proposed actions, keyless and deterministic.

> 💳 **Analogy**
>
> It is a **spending limit on a company card**. An employee can buy coffee and taxis without asking anyone - small, reversible, routine. But the moment a purchase crosses the limit, or touches a restricted category, it routes to a manager for sign-off. Nobody approves every coffee; everybody approves the big irreversible spend. The limit and the category list *are* the control - exactly like the risky-action set in an approval gate.

The agent proposes: search, process_refund, send_email, delete. Under a risk gate, how many pause for a human?

**📝 `01_approval_gate.py`** - *runnable*

In [ ]:
# APPROVAL GATES - the risk boundary. Read-only actions auto-execute; actions that
# spend money, send messages, or delete data PAUSE for a human to approve. The list of
# risky actions IS the safety mechanism. We script the "agent" so this runs with no key.

REQUIRES_APPROVAL = {"send_email", "process_refund", "delete", "purchase", "book_course"}

def gate(action):
    if action["tool"] in REQUIRES_APPROVAL:
        return "PAUSE (needs approval)"     # irreversible -> a human must sign off
    return "AUTO (read-only, safe)"         # read-only -> execute immediately

# The agent proposes these actions (tool + args):
proposed = [
    {"tool": "search",         "args": {"q": "course price"}},
    {"tool": "process_refund", "args": {"order": "12345", "amount": 50000}},
    {"tool": "send_email",     "args": {"to": "student@test.com"}},
    {"tool": "delete",         "args": {"record": "user-9"}},
]
print("Every proposed action passes the approval gate first:")
pending = []
for a in proposed:
    verdict = gate(a)
    print(f"  {a['tool']:14} -> {verdict}")
    if verdict.startswith("PAUSE"):
        pending.append(a)

# The human resolves the paused actions:
print(f"\n{len(pending)} actions paused for a human.")
print("  human APPROVES process_refund -> executed")
print("  human REJECTS  delete          -> discarded")
print("Read-only actions never waited; only the irreversible ones needed a person.")
# Output:
# Every proposed action passes the approval gate first:
#   search         -> AUTO (read-only, safe)
#   process_refund -> PAUSE (needs approval)
#   send_email     -> PAUSE (needs approval)
#   delete         -> PAUSE (needs approval)
#
# 3 actions paused for a human.
#   human APPROVES process_refund -> executed
#   human REJECTS  delete          -> discarded
# Read-only actions never waited; only the irreversible ones needed a person.

- `REQUIRES_APPROVAL` is the risky-action set - the safety boundary itself, written as data.
- `gate()` returns AUTO for read-only actions and PAUSE for anything irreversible.
- The read-only `search` runs immediately; `process_refund`, `send_email`, and `delete` all pause.
- A human then resolves the paused set: approve the refund, reject the delete - only the irreversible actions ever waited.

#### 💡 What just happened

⚡ What just happened? An approval gate classifies each proposed action by risk: read-only auto-executes, irreversible pauses for human sign-off. Tradeoff: you must curate the risky-action set (too broad floods the human, too narrow ships a mistake), but that set is the single most important safety control in an agent. Every other pattern in this lesson refines this boundary.

- Proposed actions flow through a risk gate
- Read-only auto-executes (green); money/delete/send pause for approval (amber); approve or reject

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Confidence Routing: Auto When Sure, Escalate When Not

### Confidence Routing: Auto When Sure, Escalate When Not

The agent rates its own confidence; above a threshold it acts, below it hands off to a human. The catch: raw logprobs lie, so use ensemble agreement instead.

The approval gate routes by an action’s *risk*. **Confidence routing** routes by the agent’s *certainty*: above a threshold it answers on its own; below it, it **escalates to a human**. The trap is choosing the confidence signal. The obvious one - the model’s raw token **logprobs** - is **systematically overconfident and poorly calibrated**, so a genuinely unsure answer can still score high and skip the human it needed. More robust signals are **verbalized self-assessment** (ask the model to rate itself) and, better, **ensemble agreement** (sample several answers; high agreement means real confidence). And remember that in a chain, **uncertainty compounds** - three steps each about 90 percent confident chain to roughly 73 percent. The block routes four queries by ensemble agreement against a threshold.

> 🎓 **Analogy**
>
> It is a **junior who knows when to ask the boss**. A good junior handles the routine tickets alone and escalates the ones they are unsure about - the value is entirely in the *calibration* of when to ask. A junior who is loudly certain about everything (the raw-logprob junior) is dangerous: they never escalate the case they should have. You want the one whose confidence actually tracks whether they are right.

Four queries, threshold 0.8, by ensemble agreement: [1.00], [0.40], [0.90], [0.20]. How many escalate to a human?

**📝 `02_confidence_router.py`** - *runnable*

In [ ]:
# CONFIDENCE ROUTING - auto-act when sure, escalate when not. The agent rates its own
# confidence; above the threshold it answers, below it hands off to a human. The catch:
# raw logprobs are OVERCONFIDENT, so a robust signal uses ensemble agreement instead.

THRESHOLD = 0.80

# Scripted (query, ensemble_agreement) - the fraction of sampled answers that agreed.
# High agreement across samples = high real confidence (more honest than a raw logprob).
queries = [
    ("How much does the course cost?",              1.00),   # all 5 samples agree
    ("Can I use the certificate for a US visa?",    0.40),   # samples disagree -> unsure
    ("What is the refund policy?",                  0.90),   # strong agreement
    ("Will this replace my data-science job?",      0.20),   # opinion -> low agreement
]

def route(agreement):
    return "AUTO-ANSWER" if agreement >= THRESHOLD else "ESCALATE to a human"

print(f"Route each query by ensemble agreement (threshold {THRESHOLD}):")
escalated = 0
for q, agreement in queries:
    action = route(agreement)
    if action.startswith("ESCALATE"):
        escalated += 1
    print(f"  [{agreement:.2f}] {q[:42]:42} -> {action}")
print(f"\n{4 - escalated} answered automatically, {escalated} escalated.")
print("Note: a raw logprob would rate ALL of these ~0.9+ (overconfident); ensemble agreement")
print("separates the sure from the unsure. In a chain, uncertainty compounds: 3 x 0.9 = 0.73.")
# Output:
# Route each query by ensemble agreement (threshold 0.8):
#   [1.00] How much does the course cost?             -> AUTO-ANSWER
#   [0.40] Can I use the certificate for a US visa?   -> ESCALATE to a human
#   [0.90] What is the refund policy?                 -> AUTO-ANSWER
#   [0.20] Will this replace my data-science job?     -> ESCALATE to a human
#
# 2 answered automatically, 2 escalated.
# Note: a raw logprob would rate ALL of these ~0.9+ (overconfident); ensemble agreement
# separates the sure from the unsure. In a chain, uncertainty compounds: 3 x 0.9 = 0.73.

- Each query carries an `agreement` score - the fraction of sampled answers that agreed (a calibrated stand-in for confidence).
- `route()` auto-answers at or above the threshold (0.8) and escalates below it.
- Two queries auto-answer; the visa question (0.40) and the opinion (0.20) escalate to a human.
- The note is the load-bearing lesson: a raw logprob would rate all of these high; ensemble agreement separates sure from unsure, and uncertainty compounds down a chain.

#### 💡 What just happened

⚡ What just happened? Confidence routing auto-acts above a threshold and escalates below it. Tradeoff: the threshold is domain-specific (raise it for finance and healthcare) and the confidence SIGNAL matters more than the threshold - raw logprobs are overconfident, so use ensemble agreement or verbalized confidence. Deep calibration (conformal prediction, temperature scaling) is a Module 14 topic.

- A confidence dial against a threshold: above -> auto, below -> escalate
- A logprob-vs-ensemble toggle shows how raw logprobs read overconfident

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: The Interrupt/Resume Primitive: The Re-Run Trap

### The Interrupt/Resume Primitive: The Re-Run Trap

A durable HITL pause suspends the graph and resumes later. The trap: the whole node RE-RUNS on resume, so a side effect before the pause fires twice unless it is idempotent.

Approval gates and confidence routing decide *when* to pause. This step is *how* a production pause actually works - and it hides the single most common HITL bug. In **LangGraph**, `interrupt(payload)` suspends the graph at a checkpoint and `Command(resume=value)` resumes it. Here is the trap: on resume, **the entire node re-executes from the top**, and `interrupt()` returns the resume value on that *second* pass. So any **side effect before the interrupt runs twice** - the welcome email sends again, the row inserts again - unless it is **idempotent**. This is 11.9’s idempotency rule, now load-bearing for HITL. The block runs a buggy node whose pre-interrupt side effect fires on both passes, then a guarded node that fires once.

> 🔄 **Analogy**
>
> It is a **web form that re-submits the whole page when you fix one field**. You fill in a long form, it flags one bad field, you correct it and hit submit - and if the page was built naively, it charges your card *again* because the whole request re-ran, not just the fixed field. The interrupt/resume node is that page: resuming re-runs everything above the pause. An idempotency key is the “we already processed this order” guard that makes the re-submit harmless.

A node appends a welcome email, then interrupt()s. You resume with Command(resume=True). How many emails were sent?

**📝 `03_interrupt_rerun.py`** - *runnable*

In [ ]:
# THE INTERRUPT / RESUME TRAP - a durable HITL pause re-runs the WHOLE node on resume.
# LangGraph interrupt(payload) suspends; Command(resume=value) resumes by RE-RUNNING the
# node from the top, and interrupt() returns the value on that second pass. So a side effect
# BEFORE the interrupt runs TWICE unless it is idempotent (the 11.9 rule, applied to HITL).

# --- BUGGY node: a side effect before interrupt() runs on BOTH passes ---
emails_buggy = []
def node_buggy(resume=None):
    emails_buggy.append("welcome email")          # NOT idempotent - fires every pass
    if resume is None:
        return "INTERRUPT: awaiting approval"      # pass 1: suspend
    return f"executed (approved={resume})"         # pass 2: interrupt() returned the value

node_buggy(resume=None)                            # first run -> suspend
node_buggy(resume=True)                            # Command(resume=True) -> re-runs the node
print("BUGGY node: emails sent =", len(emails_buggy), "(the welcome email went out TWICE)")

# --- SAFE node: guard the side effect with an idempotency key ---
emails_safe, seen = [], set()
def node_safe(run_id, resume=None):
    if run_id not in seen:                         # idempotency guard (from 11.9)
        seen.add(run_id); emails_safe.append("welcome email")
    if resume is None:
        return "INTERRUPT: awaiting approval"
    return f"executed (approved={resume})"

node_safe("run-1", resume=None)
node_safe("run-1", resume=True)
print("SAFE  node: emails sent =", len(emails_safe), "(guarded, so it fired ONCE)")
print("Rule: everything before interrupt() must be idempotent, because the node re-runs on resume.")
# Output:
# BUGGY node: emails sent = 2 (the welcome email went out TWICE)
# SAFE  node: emails sent = 1 (guarded, so it fired ONCE)
# Rule: everything before interrupt() must be idempotent, because the node re-runs on resume.

- `node_buggy` appends the email BEFORE the interrupt, with no guard - so it runs on pass 1 (suspend) and again on pass 2 (resume).
- Result: **emails sent = 2**. The whole node re-ran; the side effect was not idempotent.
- `node_safe` guards the append with a `seen` set keyed by `run_id` - the 11.9 idempotency pattern.
- Result: **emails sent = 1**. Everything before `interrupt()` must be idempotent, because the node re-runs on resume.

#### 💡 What just happened

⚡ What just happened? The durable interrupt/resume primitive re-runs the whole node on resume; interrupt() returns the resume value on the second pass. Tradeoff / the trap: it is the cheapest way to pause for hours without burning compute, but every pre-interrupt side effect must be idempotent or it double-fires. Never wrap interrupt() in try/except and never reorder interrupts - they are matched by position.

- A node runs, hits interrupt(), suspends; resume RE-RUNS the whole node
- A non-idempotent side effect fires twice (bug) vs guarded once

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Escalation: Route to the Right Human, With Context

### Escalation: Route to the Right Human, With Context

A tiered system: the AI handles what it can, an L1 reviewer takes low-confidence cases, an L2 expert takes the hard ones, a kill switch stops everything - and each handoff carries context.

Escalation is what happens *after* the agent decides to hand off. A flat “send it to a human” wastes the expensive humans on easy cases and starves the hard ones. Production escalation is **tiered**: the **AI** auto-handles the confident, low-risk cases; an **L1 reviewer** takes the low-confidence ones; an **L2 expert** takes the high-risk domain calls; and a **kill switch** halts everything on a critical failure. Just as important as *where* it routes is *what it carries*: every escalation should hand over a **context bundle** - the conversation history, the confidence score, and a suggested action - so the reviewer starts at line ten, not line one. And after an SLA timeout, you auto-**escalate**, never auto-approve a critical path. The block routes four cases by risk and confidence, each with its bundle.

> 🏥 **Analogy**
>
> It is a **hospital triage nurse**. Not every patient sees a surgeon, and not everyone waits in the same line. The nurse routes a scraped knee to a general clinician, a chest pain to a cardiologist, and a crash victim straight to the trauma team - and hands each one a chart, so the specialist is not starting from ‘what happened?’. Tiered routing plus the chart is exactly escalation with a context bundle.

Cases: (low risk, conf 0.95), (low, 0.55), (high, 0.90), (high + critical). Where does the critical one route?

**📝 `04_escalation_route.py`** - *runnable*

In [ ]:
# ESCALATION - route to the right human, WITH context. A tiered system: the AI handles
# what it can, an L1 reviewer takes low-confidence/low-risk, an L2 expert takes the hard
# cases, and a kill switch stops everything on a critical failure. Each handoff carries a
# context bundle so the human starts informed, not blind.

def route(risk, confidence, critical=False):
    if critical:                       return "KILL SWITCH (halt all + notify leadership)"
    if risk == "high":                 return "L2 EXPERT (domain decision)"
    if confidence < 0.80:              return "L1 REVIEWER (low confidence)"
    return "AI (auto-handle)"

def context_bundle(case):
    # Pass history + score + a suggested action so the reviewer starts at line 10, not line 1.
    return {"history": case["history"], "confidence": case["confidence"],
            "suggested": case["suggested"]}

cases = [
    {"id": 1, "risk": "low",  "confidence": 0.95, "critical": False, "history": "...", "suggested": "answer"},
    {"id": 2, "risk": "low",  "confidence": 0.55, "critical": False, "history": "...", "suggested": "clarify"},
    {"id": 3, "risk": "high", "confidence": 0.90, "critical": False, "history": "...", "suggested": "refund?"},
    {"id": 4, "risk": "high", "confidence": 0.99, "critical": True,  "history": "...", "suggested": "STOP"},
]
print("Route each case to the right tier, with context:")
for c in cases:
    tier = route(c["risk"], c["confidence"], c["critical"])
    bundle = context_bundle(c)
    print(f"  case {c['id']}: risk={c['risk']:4} conf={c['confidence']:.2f} -> {tier}")
print("\nEach escalation carries {history, confidence, suggested} so the human is not starting blind.")
# Output:
# Route each case to the right tier, with context:
#   case 1: risk=low  conf=0.95 -> AI (auto-handle)
#   case 2: risk=low  conf=0.55 -> L1 REVIEWER (low confidence)
#   case 3: risk=high conf=0.90 -> L2 EXPERT (domain decision)
#   case 4: risk=high conf=0.99 -> KILL SWITCH (halt all + notify leadership)
#
# Each escalation carries {history, confidence, suggested} so the human is not starting blind.

- `route()` checks in priority order: a critical flag trips the KILL SWITCH first, then high risk goes to L2, then low confidence to L1, else the AI auto-handles.
- Case 1 (low risk, high confidence) auto-handles; case 2 (low confidence) goes to L1.
- Case 3 (high risk) goes to the L2 expert; case 4 (critical) trips the kill switch.
- `context_bundle()` attaches history, confidence, and a suggested action to every escalation - the reviewer starts informed, not blind.

#### 💡 What just happened

⚡ What just happened? Escalation routes each case to the right tier (AI / L1 / L2 / kill switch) by risk and confidence, and carries a context bundle so the human starts informed. Tradeoff: tiers add routing logic and an on-call rota, but they put the expensive humans only on the cases that need them. On an SLA timeout, escalate up, never auto-approve.

- An action routes AI -> L1 -> L2 -> kill switch by risk and confidence
- Each escalation carries a context bundle so the reviewer starts informed

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Escalation Fatigue: The Silent Killer

### Escalation Fatigue: The Silent Killer

Too many escalations flood the reviewer, who then rubber-stamps - so the safety net collapses. The effective catch rate FALLS as the escalation rate rises. The sweet spot is roughly 10-15 percent.

Here is the failure mode that kills more HITL systems than missed approvals ever do: **escalation fatigue**. The intuition “more human review is safer” is *wrong past a point*. A reviewer who sees a handful of thoughtful escalations reads each one carefully. A reviewer drowning in a thousand a day stops reading and starts clicking “approve” - and now the safety mechanism is *worse* than useless, because a human has formally signed off on mistakes nobody looked at. The **effective catch rate is not constant**; it *falls* as the escalation rate climbs. The practitioner sweet spot is roughly **10 to 15 percent**: escalate only the high-impact, genuinely-uncertain cases, and gather context so each one is fast to judge. The block models how the catch rate degrades as the flood grows.

> 🔔 **Analogy**
>
> It is a **smoke alarm you disable because it cries wolf**. An alarm that goes off every time you make toast does not make you safer - within a week you have pulled the battery, and now it is silent for the real fire too. An over-escalating agent is that alarm: cry wolf often enough and the human tunes you out entirely. The goal is an alarm that only sounds when it matters, so it is still trusted when it counts.

You push the escalation rate from about 10 percent up to 60 percent. What happens to the effective catch rate?

**📝 `05_fatigue.py`** - *runnable*

In [ ]:
# ESCALATION FATIGUE - the SILENT KILLER of HITL. Too many escalations flood the reviewer,
# who then rubber-stamps, and the safety net collapses. Effective catch rate is NOT constant:
# it falls as the escalation rate rises. The practitioner sweet spot is roughly 10-15%.

def effective_catch_rate(escalation_rate):
    # A sharp reviewer catches ~95% of real problems; a flooded one rubber-stamps.
    # Attention degrades as the queue grows past a sustainable load.
    if escalation_rate <= 0.15:   attention = 0.95      # sustainable: reviewers stay sharp
    elif escalation_rate <= 0.30: attention = 0.80      # manageable but slipping
    elif escalation_rate <= 0.45: attention = 0.60
    else:                         attention = 0.40      # flooded -> rubber-stamping
    return attention

print("How the escalation rate changes the effective catch rate:")
for rate in [0.10, 0.20, 0.35, 0.60]:
    catch = effective_catch_rate(rate)
    tag = "sweet spot" if rate <= 0.15 else ("manageable" if rate <= 0.30 else "FATIGUE")
    print(f"  escalate {int(rate*100):>3}%  -> reviewer catches ~{int(catch*100)}% of real issues   ({tag})")
print("\nMore escalation is NOT safer past ~15%: flooded reviewers rubber-stamp and the net collapses.")
print("Escalate only the high-impact/uncertain cases; gather context first; keep the rate near 10-15%.")
# Output:
# How the escalation rate changes the effective catch rate:
#   escalate  10%  -> reviewer catches ~95% of real issues   (sweet spot)
#   escalate  20%  -> reviewer catches ~80% of real issues   (manageable)
#   escalate  35%  -> reviewer catches ~60% of real issues   (FATIGUE)
#   escalate  60%  -> reviewer catches ~40% of real issues   (FATIGUE)
#
# More escalation is NOT safer past ~15%: flooded reviewers rubber-stamp and the net collapses.
# Escalate only the high-impact/uncertain cases; gather context first; keep the rate near 10-15%.

- `effective_catch_rate()` models reviewer attention as a function of load, not a constant.
- At about 10 percent escalation, reviewers stay sharp and catch roughly 95 percent of real issues - the sweet spot.
- As the rate climbs to 35 and 60 percent, attention degrades and the catch rate falls toward 40 percent - rubber-stamping.
- The counter-intuitive result: pushing MORE to humans past about 15 percent makes the system LESS safe, not more.

#### 💡 What just happened

⚡ What just happened? Escalation fatigue is the primary HITL failure mode: too many escalations flood reviewers into rubber-stamping, and the effective catch rate collapses. Tradeoff / the whole point: more human review is only safer up to roughly 10-15 percent; beyond that you must escalate LESS and with better context, not more. Measure your escalation rate the way you measure error rate.

- Slide the escalation rate; reviewer attention and catch rate fall as it climbs
- The 10-15 percent sweet spot vs the 60 percent rubber-stamp collapse

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Progressive Autonomy: Earn Trust, Revoke It Instantly

### Progressive Autonomy: Earn Trust, Revoke It Instantly

Start at full review (Audit), advance to mostly-auto (Assist), then sampling-only (Automate) - but only when hard gates pass, and always reversibly: a critical error snaps the tier straight back.

You do not hand a new agent the keys on day one, and you do not lock it at full-review forever. **Progressive autonomy** is the graduated middle path: the agent **earns** trust in stages and can have it **revoked instantly**. Start at **Audit** - a human reviews 100 percent of outputs. When it clears a hard gate (high accuracy over a large sample, a low override rate, zero critical errors in the window), advance to **Assist** - auto-run, human reviews only the risky ones. Clear the gate again and reach **Automate** - sampling-only spot checks. The two rules that make this safe: advancement is **gated on evidence**, not vibes or a calendar; and it is **always reversible** - a single critical error snaps the tier straight back to Audit, with no code change. The block runs the trust FSM through four review windows, including a rollback.

> 🚘 **Analogy**
>
> It is a **graduated driver’s licence**. A new driver starts with a learner permit (supervised at all times), earns a provisional licence (drive alone, but with restrictions), and only then a full one. Each step is *earned* by a clean record over time - and a serious violation sends you straight back a stage. You would not hand a fresh learner an unrestricted licence, and you would not let a reckless one keep it. Agent autonomy graduates the same way.

An agent at Automate makes ONE critical error. In a reversible trust system, what tier is it on next?

**📝 `06_trust_tier.py`** - *runnable*

In [ ]:
# PROGRESSIVE AUTONOMY - earn trust in stages, revoke it instantly. Start at full review
# (AUDIT), advance to mostly-auto (ASSIST), then sampling-only (AUTOMATE) - but only when
# hard gates pass, and ALWAYS reversibly: a critical error snaps the tier straight back.

TIERS = ["AUDIT", "ASSIST", "AUTOMATE"]           # 100% review -> risky-only -> sampling-only

def next_tier(current, accuracy, override_rate, critical_error):
    if critical_error:
        return "AUDIT"                             # instant rollback to full review
    i = TIERS.index(current)
    passes = accuracy >= 0.98 and override_rate < 0.02   # the advancement gate
    if passes and i < len(TIERS) - 1:
        return TIERS[i + 1]                        # advance one tier
    return current                                 # hold

# A stream of monthly review windows (accuracy, override_rate, critical_error):
windows = [
    (0.985, 0.010, False),   # good -> advance
    (0.990, 0.008, False),   # good -> advance
    (0.999, 0.001, True),    # a critical error -> rollback
    (0.985, 0.015, False),   # good -> advance
]
tier = "AUDIT"
print("Trust advances on the gate (accuracy >=98%, override <2%) and rolls back on a critical error:")
for i, (acc, ov, crit) in enumerate(windows, 1):
    tier = next_tier(tier, acc, ov, crit)
    note = "CRITICAL ERROR -> rollback" if crit else "gate passed -> advance"
    print(f"  window {i}: acc={acc:.3f} override={ov:.3f} {'crit' if crit else '    '} -> {tier:8} ({note})")
print("\nAutonomy is earned incrementally and revoked instantly; the progression is always reversible.")
# Output:
# Trust advances on the gate (accuracy >=98%, override <2%) and rolls back on a critical error:
#   window 1: acc=0.985 override=0.010      -> ASSIST   (gate passed -> advance)
#   window 2: acc=0.990 override=0.008      -> AUTOMATE (gate passed -> advance)
#   window 3: acc=0.999 override=0.001 crit -> AUDIT    (CRITICAL ERROR -> rollback)
#   window 4: acc=0.985 override=0.015      -> ASSIST   (gate passed -> advance)
#
# Autonomy is earned incrementally and revoked instantly; the progression is always reversible.

- `TIERS` is the ladder: Audit (100 percent review) -> Assist (risky-only) -> Automate (sampling).
- `next_tier()` advances only when the gate passes (accuracy at or above 98 percent AND override below 2 percent).
- Windows 1 and 2 pass the gate, so the tier climbs Audit -> Assist -> Automate.
- Window 3 has a critical error -> instant rollback to Audit, regardless of its otherwise-perfect metrics; window 4 re-earns Assist.

#### 💡 What just happened

⚡ What just happened? Progressive autonomy earns trust in evidence-gated stages (Audit -> Assist -> Automate) and revokes it instantly on a critical error. Tradeoff: you carry review cost longest at the start and must define the gates and the rollback trigger, but you never grant more autonomy than the agent has demonstrably earned, and a bad day is recoverable in one step.

- A trust meter climbs Audit -> Assist -> Automate as the metric gates pass
- A critical violation snaps it straight back to Audit

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: The Framework Landscape + When HITL Is Mandatory

### The Framework Landscape + When HITL Is Mandatory

Pick the HITL primitive by the need - crash-durable, framework-native, or a quick local gate - and know that for high-risk AI, human oversight is legally required (EU AI Act Article 14).

Finally, the judgment call: which primitive implements the pause. **Crash-durable** approval that must survive a restart → **Temporal signals** (the 11.9 durability guarantee). A **framework-native** pause inside an agent run → **OpenAI**’s `needsApproval` (the run pauses and returns pending interruptions), **Anthropic**’s `canUseTool` callback with permission modes, or **LangGraph**’s `interrupt()`. A **quick local** check → your own manual gate (Step 1). And this is not only an engineering choice: for **high-risk AI, human oversight is legally required**. The **EU AI Act Article 14** mandates that high-risk systems let a human *monitor, interpret, intervene in, and override* their decisions, with explicit awareness of automation bias; remote biometric identification needs verification by at least two people. Those obligations phase in through **August 2026**. The runnable block routes needs to primitives; the illustrative block shows the two framework APIs.

> 🔒 **Analogy**
>
> It is **choosing the right lock for the door**. A garden gate takes a simple latch (a local gate); a front door takes a deadbolt (a framework-native pause); a bank vault takes a time-lock that holds even through a power cut (a durable Temporal signal). You match the lock to what is behind the door - and for some doors, building regulations *require* a certain lock (the EU AI Act). You would not put a garden latch on a vault.

You need an approval that must survive the server restarting mid-wait. Which primitive fits?

**📝 `07_hitl_router.py`** - *runnable*

In [ ]:
# THE FRAMEWORK LANDSCAPE - pick the HITL primitive by the need. Crash-durable approval
# (survives a restart) -> Temporal signals. A framework-native pause -> OpenAI needsApproval
# or LangGraph interrupt(). A quick local check -> a manual gate. And for high-risk AI, human
# oversight is LEGALLY REQUIRED (EU AI Act Article 14), not optional.

def choose(needs_crash_durability, in_a_framework, high_risk_regulated):
    primitive = ("Temporal durable signals" if needs_crash_durability
                 else "OpenAI needsApproval / LangGraph interrupt()" if in_a_framework
                 else "a manual approval gate (your own code)")
    mandate = " + EU AI Act Article 14 human-oversight REQUIRED" if high_risk_regulated else ""
    return primitive + mandate

cases = [
    ("8-hour approval that must survive a restart", dict(needs_crash_durability=True,  in_a_framework=True,  high_risk_regulated=False)),
    ("pause a tool call inside an agent run",       dict(needs_crash_durability=False, in_a_framework=True,  high_risk_regulated=False)),
    ("a quick script that gates one action",        dict(needs_crash_durability=False, in_a_framework=False, high_risk_regulated=False)),
    ("a credit-decisioning agent (regulated)",      dict(needs_crash_durability=True,  in_a_framework=True,  high_risk_regulated=True)),
]
print("Route each need to the right HITL primitive:")
for name, props in cases:
    print(f"  {name:44} -> {choose(**props)}")
print("\nDurable approval -> Temporal; framework pause -> needsApproval / interrupt; and high-risk AI")
print("MUST allow a human to monitor, intervene, and override (EU AI Act Article 14).")
# Output:
# Route each need to the right HITL primitive:
#   8-hour approval that must survive a restart  -> Temporal durable signals
#   pause a tool call inside an agent run        -> OpenAI needsApproval / LangGraph interrupt()
#   a quick script that gates one action         -> a manual approval gate (your own code)
#   a credit-decisioning agent (regulated)       -> Temporal durable signals + EU AI Act Article 14 human-oversight REQUIRED
#
# Durable approval -> Temporal; framework pause -> needsApproval / interrupt; and high-risk AI
# MUST allow a human to monitor, intervene, and override (EU AI Act Article 14).

- `choose()` routes by the need: crash-durability -> Temporal signals; a framework pause -> OpenAI needsApproval / LangGraph interrupt; else a manual gate.
- The regulated credit-decisioning case adds the EU AI Act Article 14 human-oversight mandate on top of the primitive.
- High-risk AI must let a human monitor, intervene, and override - that is architecture you are required to build, not optional polish.
- The illustrative block below shows the two framework APIs concretely.

**📝 `07b_framework_hitl.py`** - *illustrative*

In [ ]:
# FRAMEWORK HITL - the current 2026 APIs (illustrative minimal example).
# OpenAI Agents SDK: mark a tool needs_approval; the run PAUSES and returns pending
# interruptions you approve or reject. Anthropic Claude Agent SDK: a can_use_tool
# callback returns allow/deny per tool call, shaped by permission modes.

# --- OpenAI Agents SDK (Python) ---
from agents import Agent, function_tool

@function_tool(needs_approval=True)          # money-moving tool -> the run pauses here
def process_refund(order_id: str, amount: int) -> str:
    return f"refunded {amount} on {order_id}"

# result = await Runner.run(agent, "refund order 123")
# if result.interruptions:                   # the run stopped, awaiting a human
#     result.interruptions[0].approve()      # or .reject() - your UI decides
#     result = await Runner.run(agent, result.to_input_list())

# --- Anthropic Claude Agent SDK (Python) ---
async def can_use_tool(tool_name, tool_input, context):
    if tool_name in {"Bash", "process_refund", "delete"}:
        return {"behavior": "deny", "message": "needs human approval"}   # gate the risky ones
    return {"behavior": "allow", "updatedInput": tool_input}             # auto-allow read-only

# ClaudeSDKClient(options=ClaudeAgentOptions(can_use_tool=can_use_tool,
#     permission_mode="default"))  # or bypassPermissions / acceptEdits / plan / dontAsk
# A PreToolUse hook runs before EVERY step and its deny wins even under bypassPermissions.
# Output: (illustrative minimal example - needs `pip install openai-agents` / `claude-agent-sdk`.)
# needs_approval pauses the run for sign-off; can_use_tool denies the risky tools and allows read-only.

- OpenAI: `@function_tool(needs_approval=True)` makes the run PAUSE and return pending `interruptions` you `.approve()` or `.reject()`.
- Anthropic: a `can_use_tool` callback returns `allow`/`deny` per tool call - deny the risky tools, allow read-only.
- **permission modes** shape it: `bypassPermissions` / `acceptEdits` / `plan` / `dontAsk` with an allow-list.
- A `PreToolUse` hook runs before every step and its deny wins even under `bypassPermissions` - the strongest gate.

#### 💡 What just happened

⚡ What just happened? Pick the HITL primitive by the need: Temporal signals for crash-durable approval, OpenAI needsApproval / Anthropic canUseTool / LangGraph interrupt for a framework-native pause, a manual gate for quick local checks. Tradeoff: durable primitives cost an orchestrator; framework ones are simpler but do not survive a crash on their own. And for high-risk AI, HITL is not a choice - EU AI Act Article 14 requires it.

- Pick a need: durable? framework-native? terminal-local?
- See the primitive (Temporal / OpenAI needsApproval / LangGraph interrupt) + the compliance trigger

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture
> Human-in-the-loop is one decision drawn precisely. An **approval gate** puts a human on the irreversible actions and auto-runs the rest (Step 1); **confidence routing** puts a human on the uncertain ones, using ensemble agreement because logprobs lie (Step 2). The durable **interrupt/resume** primitive makes that pause survive hours without burning compute - as long as pre-interrupt side effects are idempotent, because the node re-runs (Step 3). When it does escalate, it routes to the right **tier with context** (Step 4), and it watches the **escalation rate** so reviewers never flood into rubber-stamping (Step 5). Trust is **earned in reversible stages**, Audit to Assist to Automate (Step 6). And you implement the pause with the right **primitive** - Temporal, a framework, or a local gate - knowing that for high-risk AI, human oversight is **legally required** (Step 7).

| HITL primitive | Mechanism | Durability | Best for |
|---|---|---|---|
| OpenAI needsApproval | tool flagged; run returns pending interruptions | in-run (you persist) | a framework-native pause you approve/reject |
| Anthropic canUseTool | allow/deny callback + permission modes + PreToolUse hook | in-run (you persist) | per-tool policy for a coding/computer agent |
| LangGraph interrupt() | suspend at checkpoint; Command(resume=) re-runs the node | durable (with a checkpointer) | graph agents; resume days later by thread_id |
| Temporal signals | wait_condition() + handle.signal() | survives a crash or deploy | long, crash-durable approvals |

> 📦 **Info**
>
> ➡️ Where this goes nextYou can now put a human exactly where their judgment changes the outcome, and nowhere else. We come back to **evaluating and monitoring** these agents - measuring the escalation rate and override rate you tune here - in Lesson 11.11. **Confidence calibration** (conformal prediction, temperature scaling) is in Module 14, and **India compliance** (RBI FREE-AI, SEBI, DPDP) plus the reviewer-cost economics are in Lesson 17.3 and Module 15. The primary references are the [OpenAI Agents SDK HITL guide](https://openai.github.io/openai-agents-js/guides/human-in-the-loop/), the [Claude Agent SDK permissions docs](https://code.claude.com/docs/en/agent-sdk/permissions), the Anthropic [auto-mode engineering post](https://www.anthropic.com/engineering/claude-code-auto-mode), the open-source [claude-agent-sdk-python](https://github.com/anthropics/claude-agent-sdk-python) repo, and [EU AI Act Article 14](https://artificialintelligenceact.eu/article/14/).

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Human-in-the-Loop**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-11_10.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-11.10.html` - regenerate this notebook whenever the source HTML is updated.*
